# ST-OMR Meter V5-2Y — Lexicographic Parameter Stability Audit

Single-run TRAIN-only diagnostic pinned to exact CI-green commit `18e23ed2c25e50db03f41db70259db3fd74e224a`. It keeps the exact V5-2X functional-drift bound plus fixed numerical slack and fits one temporary min-Linf delta-weight witness per specialist. It does not train or mutate a model, save witness values, tune bias/threshold, select a repair, or open Historical VALIDATION, First-30, V5 VAL, or FINAL_HOLDOUT.

In [ ]:
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path
import hashlib
import json
import subprocess
import sys
import time

EXPECTED_HEAD = "18e23ed2c25e50db03f41db70259db3fd74e224a"
EXPECTED_CI_RUN_ID = 32707911972
EXPECTED_SCIPY_VERSION = "1.18.0"
REPOSITORY = "khfy7wpr5p-maker/st-omr-training"
REPO_URL = f"https://github.com/{REPOSITORY}.git"
REPO = Path("/content/st-omr-training")
MYDRIVE = Path("/content/drive/MyDrive")

if not MYDRIVE.is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
DATA_ROOT = MYDRIVE / "TEST" / "METER_V2_1500_PACKAGE_AB_CLEAN"
CHECKPOINT_ROOT = MYDRIVE / "ST-OMR-METER-SPECIALISTS"
M4A_ROOT = CHECKPOINT_ROOT / "m4a-234-digit-specialist-dataset-freeze-v2"
D10_ROOT = MYDRIVE / "ST-OMR-D10" / "stage7d10-authoritative-562c8fcfabf1b41573f1ef591d88ae65335ce16a"
for name, path in {"DATA_ROOT": DATA_ROOT, "CHECKPOINT_ROOT": CHECKPOINT_ROOT, "M4A_ROOT": M4A_ROOT, "D10_ROOT": D10_ROOT}.items():
    if not path.is_dir():
        raise RuntimeError(f"{name} bulunamadi: {path}")
print("DRIVE CHECK = PASS")

if not REPO.exists():
    subprocess.check_call(["git", "clone", "--no-checkout", REPO_URL, str(REPO)])
elif not (REPO / ".git").is_dir():
    raise RuntimeError(f"REPO git repository degil: {REPO}")
remotes = subprocess.check_output(["git", "-C", str(REPO), "remote"], text=True).split()
if "origin" not in remotes:
    subprocess.check_call(["git", "-C", str(REPO), "remote", "add", "origin", REPO_URL])
else:
    subprocess.check_call(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL])
subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin", EXPECTED_HEAD, "--depth", "1"])
fetched_head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "FETCH_HEAD"], text=True).strip()
if fetched_head != EXPECTED_HEAD:
    raise RuntimeError(f"FETCH_HEAD mismatch: expected={EXPECTED_HEAD} actual={fetched_head}")
subprocess.check_call(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_HEAD])
actual_head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
if actual_head != EXPECTED_HEAD:
    raise RuntimeError(f"HEAD mismatch: {actual_head}")
if subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip():
    raise RuntimeError("Repository worktree temiz degil")
print("REPOSITORY CHECK = PASS")
print("HEAD =", actual_head)
print("CI RUN ID =", EXPECTED_CI_RUN_ID)

if metadata.version("scipy") != EXPECTED_SCIPY_VERSION:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "scipy==1.18.0"])
if metadata.version("scipy") != EXPECTED_SCIPY_VERSION:
    raise RuntimeError(f"SciPy version mismatch: {metadata.version('scipy')}")
print("SCIPY VERSION CHECK = PASS", metadata.version("scipy"))

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from st_omr_training import meter_v5_1_bbox_pilot as v51
from st_omr_training import meter_v5_2b_specialist_adaptation as v52b
from st_omr_training import meter_v5_2x_minimum_functional_logit_drift_audit_v1 as v52x
from st_omr_training import meter_v5_2y_lexicographic_parameter_stability_audit_v1 as audit
print("MODULE IMPORT = PASS")

DIGIT2_FROZEN = v52b.locate_checkpoint_by_sha_v1(CHECKPOINT_ROOT, v52b.DIGIT2_SHA256)
DIGIT3_FROZEN = v52b.locate_checkpoint_by_sha_v1(CHECKPOINT_ROOT, v52b.DIGIT3_SHA256)
ANN_DIR = DATA_ROOT / "annotations"
V52X_REPORT = ANN_DIR / v52x.REPORT_NAME
V52X_ENVELOPE = ANN_DIR / f"v5_2x_execution_envelope_{audit.V52X_IMPLEMENTATION_HEAD}.json"
REPORT_PATH = ANN_DIR / audit.REPORT_NAME
ENVELOPE_PATH = ANN_DIR / f"v5_2y_execution_envelope_{EXPECTED_HEAD}.json"
for name, path in {"V5-2X REPORT": V52X_REPORT, "V5-2X ENVELOPE": V52X_ENVELOPE}.items():
    if not path.is_file():
        raise RuntimeError(f"{name} missing: {path}")
for path in (REPORT_PATH, ENVELOPE_PATH):
    if path.exists():
        raise RuntimeError(f"Refusing overwrite/rerun: {path}")
print("EXACT INPUT BINDING = PASS")
print("OUTPUT GUARD = PASS")

required_safety = {
    "model_training": False,
    "autograd_grad_used": False,
    "backward": False,
    "optimizer_steps": 0,
    "diagnostic_linear_program_solve": True,
    "diagnostic_lexicographic_witness_fit": True,
    "diagnostic_witness_persisted": False,
    "diagnostic_witness_values_emitted": False,
    "classifier_fit_for_deployment": False,
    "checkpoint_read": True,
    "candidate_checkpoint_write": False,
    "model_parameter_mutation": False,
    "threshold_tuning": False,
    "alternative_threshold_evaluated": False,
    "bias_selection": False,
    "historical_validation_opened": False,
    "historical_validation_report_read": False,
    "historical_validation_error_examples_read": False,
    "first30_opened": False,
    "v5_validation_opened": False,
    "final_holdout_locked": True,
    "digit4_frozen": True,
    "per_example_rows_emitted": False,
    "repair_selected": False,
    "repair_training_authorized": False,
    "production_promotion": False,
}
for key, expected in required_safety.items():
    if audit.safety_boundary().get(key) != expected:
        raise RuntimeError(f"Safety boundary mismatch: {key}")
solver = audit.solver_contract()
if solver.get("library_version_matches_expected") is not True:
    raise RuntimeError("Pinned SciPy solver contract mismatch")
if solver.get("objective") != "minimum_max_absolute_delta_weight":
    raise RuntimeError("Secondary objective contract changed")
if solver.get("primary_drift_absolute_slack") != 1e-6:
    raise RuntimeError("Primary drift slack contract changed")
if solver.get("primary_functional_drift_reoptimized") is not False:
    raise RuntimeError("Primary functional objective was unexpectedly reoptimized")
if solver.get("maximum_absolute_delta_weight_minimized") is not True:
    raise RuntimeError("Secondary min-Linf objective is not active")
if solver.get("weight_l1_minimized") is not False or solver.get("weight_l2_minimized") is not False:
    raise RuntimeError("Unregistered weight objective enabled")
if solver.get("automatic_second_solver") is not False or solver.get("solver_sweep") is not False:
    raise RuntimeError("Fallback solver or sweep unexpectedly enabled")
print("DIAGNOSTIC SAFETY BOUNDARY = PASS")
print("MODEL_TRAINING=False | DIAGNOSTIC_LEXICOGRAPHIC_WITNESS_FIT=True")
print("PRIMARY_DRIFT=V5-2X+1e-6 | SECONDARY_OBJECTIVE=MIN_MAX_ABS_DELTA_WEIGHT")
print("THRESHOLDS=FROZEN | BIAS_SELECTION=False | WITNESS_SAVE=False")
print("HISTORICAL_VALIDATION=CLOSED | FIRST-30=CLOSED")
print("V5_VAL=CLOSED | FINAL_HOLDOUT=LOCKED | 4-AI=FROZEN")

started = time.time()
def progress(processed, total, phase):
    if processed == 1 or processed == total or processed % 2048 == 0:
        print(phase, f"{processed}/{total}", f"| elapsed={int(time.time() - started)}s")

report = audit.run_lexicographic_parameter_stability_audit_v1(
    DATA_ROOT,
    m4a_root=M4A_ROOT,
    d10_root=D10_ROOT,
    digit2_frozen=DIGIT2_FROZEN,
    digit3_frozen=DIGIT3_FROZEN,
    v5_2x_report=V52X_REPORT,
    v5_2x_envelope=V52X_ENVELOPE,
    progress=progress,
)
if not REPORT_PATH.is_file():
    raise RuntimeError(f"Audit report missing: {REPORT_PATH}")
report_bytes = REPORT_PATH.read_bytes()
saved_report = json.loads(report_bytes.decode("utf-8"))
if saved_report != report:
    raise RuntimeError("Saved report mismatch")
for key, expected in required_safety.items():
    if report.get(key) != expected:
        raise RuntimeError(f"Saved report safety mismatch: {key}")
allowed_claims = {
    "WITNESS_VERIFIED",
    "SOLVER_REPORTED_INFEASIBLE_NOT_FORMAL_PROOF",
    "UNPROVEN_SOLVER_DID_NOT_RETURN_A_USABLE_STATUS",
    "UNPROVEN_WITNESS_RESIDUAL_FAILED",
}
for digit in ("2", "3"):
    item = report["per_specialist"][digit]["lexicographic_parameter_stability"]
    diagnosis = report["per_specialist"][digit]["path_diagnosis"]
    claim = item.get("witness_claim")
    if claim not in allowed_claims:
        raise RuntimeError(f"{digit}-AI unknown witness claim: {claim}")
    if item.get("witness_values_emitted") is not False:
        raise RuntimeError(f"{digit}-AI emitted witness values")
    if item.get("witness_persisted") is not False:
        raise RuntimeError(f"{digit}-AI persisted a witness")
    if item.get("maximum_absolute_delta_weight_minimized") is not True:
        raise RuntimeError(f"{digit}-AI min-Linf objective flag changed")
    if item.get("weight_l1_minimized") is not False or item.get("weight_l2_minimized") is not False:
        raise RuntimeError(f"{digit}-AI overclaimed L1/L2 minimization")
    if diagnosis.get("repair_selected") is not False:
        raise RuntimeError(f"{digit}-AI unexpectedly selected a repair")
    if diagnosis.get("generalization_proven") is not False:
        raise RuntimeError(f"{digit}-AI overclaimed generalization")
    if diagnosis.get("deployment_stability_proven") is not False:
        raise RuntimeError(f"{digit}-AI overclaimed deployment stability")
    if claim == "WITNESS_VERIFIED":
        if item.get("parameter_stability_witness_verified") is not True:
            raise RuntimeError(f"{digit}-AI witness verification flag mismatch")
        if item.get("secondary_optimality_claim") != "SOLVER_REPORTED_OPTIMAL_NOT_FORMAL_PROOF":
            raise RuntimeError(f"{digit}-AI secondary optimality claim mismatch")
        for key in ("v5_constraint_violations", "historical_retention_constraint_violations", "historical_drift_cap_violations", "parameter_bound_violations"):
            if item.get(key) != 0:
                raise RuntimeError(f"{digit}-AI residual violation: {key}")
        if item.get("functional_delta_identity_verified") is not True:
            raise RuntimeError(f"{digit}-AI functional identity failed")
        if item.get("primary_optimum_consistency_verified") is not True:
            raise RuntimeError(f"{digit}-AI contradicted V5-2X primary optimum")
        if item["independently_recomputed_max_absolute_historical_logit_drift"] > item["primary_drift_cap"] + audit.WITNESS_TOLERANCE:
            raise RuntimeError(f"{digit}-AI exceeded primary drift cap")
        if abs(item["minimum_max_absolute_delta_weight"] - item["independently_recomputed_max_absolute_delta_weight"]) > audit.WITNESS_TOLERANCE:
            raise RuntimeError(f"{digit}-AI secondary objective recomputation mismatch")
        if item["historical_transition_counts"].get("correct_to_wrong") != 0:
            raise RuntimeError(f"{digit}-AI historical TRAIN decision regression")
        if item["diagnostic_v5_train_metrics"].get("f1") != 1.0:
            raise RuntimeError(f"{digit}-AI V5 TRAIN is not perfect")
    elif item.get("parameter_stability_witness_verified") is not False:
        raise RuntimeError(f"{digit}-AI unverified witness marked verified")
post_run_head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
if post_run_head != EXPECTED_HEAD:
    raise RuntimeError(f"Post-run HEAD mismatch: {post_run_head}")
if subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip():
    raise RuntimeError("Repository changed during audit")
report_sha256 = hashlib.sha256(report_bytes).hexdigest()
envelope = {
    "schema": "st-omr-meter-v5-2y-exact-sha-execution-envelope-v1",
    "repository": REPOSITORY,
    "expected_head": EXPECTED_HEAD,
    "actual_head_before_run": actual_head,
    "actual_head_after_run": post_run_head,
    "ci_run_id": EXPECTED_CI_RUN_ID,
    "executed_at_utc": datetime.now(timezone.utc).isoformat(),
    "audit_report_name": audit.REPORT_NAME,
    "audit_report_sha256": report_sha256,
    "solver_contract": report["solver_contract"],
    "claims": {digit: report["per_specialist"][digit]["lexicographic_parameter_stability"]["witness_claim"] for digit in ("2", "3")},
    "primary_drift_caps": {digit: report["per_specialist"][digit]["lexicographic_parameter_stability"]["primary_drift_cap"] for digit in ("2", "3")},
    "minimum_max_absolute_delta_weight": {digit: report["per_specialist"][digit]["lexicographic_parameter_stability"].get("minimum_max_absolute_delta_weight") for digit in ("2", "3")},
    "diagnosis": {digit: report["per_specialist"][digit]["path_diagnosis"] for digit in ("2", "3")},
    "safety_boundary": {key: report[key] for key in required_safety},
}
v51._atomic_write_json(ENVELOPE_PATH, envelope)
envelope_sha256 = hashlib.sha256(ENVELOPE_PATH.read_bytes()).hexdigest()

print()
print("============================================")
print("V5-2Y LEXICOGRAPHIC PARAMETER STABILITY RESULT")
print("============================================")
for digit in ("2", "3"):
    item = report["per_specialist"][digit]["lexicographic_parameter_stability"]
    diagnosis = report["per_specialist"][digit]["path_diagnosis"]
    print()
    print(f"========== {digit}-AI ==========")
    print("SOLVER =", {key: item[key] for key in ("status", "success", "iterations", "witness_claim", "secondary_optimality_claim")})
    if item["witness_claim"] == "WITNESS_VERIFIED":
        print("V5-2X PRIMARY DRIFT BOUND =", item["primary_max_absolute_historical_logit_drift"])
        print("FIXED PRIMARY DRIFT CAP =", item["primary_drift_cap"])
        print("ACTUAL MAX ABS HISTORICAL LOGIT DRIFT =", item["independently_recomputed_max_absolute_historical_logit_drift"])
        print("MINIMUM MAX ABS DELTA WEIGHT =", item["minimum_max_absolute_delta_weight"])
        print("HISTORICAL ABS LOGIT DRIFT =", item["historical_absolute_logit_drift"])
        print("MINIMUM V5 MARGIN =", item["minimum_v5_signed_decision_margin"])
        print("MINIMUM HISTORICAL RETAINED MARGIN =", item["minimum_historical_retained_signed_decision_margin"])
        print("V5 TRANSITIONS =", item["v5_transition_counts"])
        print("HISTORICAL TRANSITIONS =", item["historical_transition_counts"])
        print("V5 TRAIN METRICS =", item["diagnostic_v5_train_metrics"])
        print("HISTORICAL TRAIN METRICS =", item["diagnostic_historical_train_metrics"])
        print("WEIGHT GEOMETRY =", item["weight_geometry"])
    print("PATH DIAGNOSIS =", diagnosis)
print()
print("EXACT SHA EXECUTION = PASS")
print("HEAD =", post_run_head)
print("REPORT =", REPORT_PATH)
print("REPORT SHA256 =", report_sha256)
print("EXECUTION ENVELOPE =", ENVELOPE_PATH)
print("ENVELOPE SHA256 =", envelope_sha256)
print("MODEL TRAINING EXECUTED = False")
print("DIAGNOSTIC LEXICOGRAPHIC WITNESS FIT = True | WITNESS SAVED = False")
print("REPAIR SELECTED = False")
print("FIRST-30 = CLOSED | V5 VAL = CLOSED | FINAL HOLDOUT = LOCKED")
